# Experiment 48 — BPTT + Structural Sparse Memory

Tests whether allocate-on-arrival / local rewiring complements gradient learning. Continuous parameters are trained by FullCE+BPTT; concept ownership and graph destinations are discrete structural updates.

Arms: `canonical_full`, `full_alloc`, `full_alloc_rewire`, `event_alloc_rewire`. The last arm detaches recurrent mass before every event.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys
repo = '/content/Sparsewalker'
if not os.path.exists(repo):
    subprocess.run(['git','clone','-b','research/active','https://github.com/hanialshater/Sparsewalker-.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','research/active'], check=True)
    subprocess.run(['git','-C',repo,'checkout','research/active'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/research/active'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',repo], check=True)
print(subprocess.check_output(['git','-C',repo,'rev-parse','--short','HEAD'], text=True).strip())

## Run
The canonical control runs first. Default is 12 screening epochs while retaining the canonical 50-epoch LR schedule.

In [ ]:
import os, runpy, sys
repo = '/content/Sparsewalker'
script = os.path.join(repo, 'experiments', 'run_ml1m_bptt_structural_memory.py')
os.chdir(repo)
for p in [os.path.join(repo,'src'), os.path.join(repo,'experiments')]:
    if p not in sys.path: sys.path.insert(0,p)
sys.argv = [script,
    '--epochs','12',
    '--schedule-epochs','50',
    '--batch-size','128',
    '--eval-batch-size','1024',
    '--memory-lambda','0.5',
    '--max-aliases','16',
    '--max-rewires-per-batch','1024',
    '--progress-every','0',
    '--data-dir','/content/drive/MyDrive/sparsewalker_data',
    '--output-dir','/content/drive/MyDrive/sparsewalker_bptt_structural']
print('RUNNING_IN_PROCESS', ' '.join(sys.argv))
runpy.run_path(script, run_name='__main__')

In [ ]:
import json, os, pandas as pd
summary_path = '/content/drive/MyDrive/sparsewalker_bptt_structural/seed42/summary.json'
with open(summary_path) as f:
    summary = json.load(f)
df = pd.DataFrame(summary['ranking'])
display(df[[
    'arm','temporal_credit','structural_allocation','structural_rewire',
    'best_dense_NDCG@10','best_hybrid_NDCG@10','best_sparse_candidate_recall',
    'final_dense_NDCG@10','final_hybrid_NDCG@10','final_sparse_candidate_recall',
    'final_occupied_concepts','final_item_coverage','mean_positions_per_s'
]])
print(json.dumps(summary, indent=2))